# run_experiment

One experiment, end to end, from a settings file. Works on Colab and on a rented
GPU box; the only difference is which settings file you name.

Edit the first code cell and nothing else: the settings file, the branch, and the Hub repo. To run a second experiment, change
`SETTINGS_FILE` and re-run from **Run the experiment** downwards. The model stays
downloaded and the environment stays installed, so each extra run costs only its
own training time.

On a rented machine that expires, run the save cell after every experiment rather
than at the end. Everything on that box is deleted when it shuts down.

In [8]:
USE_TEST_CONFIG = False         # True: the CPU smoke settings in src/configs/test.py
SETTINGS_FILE = "configs/one_persona_minimum_threshold_b.json"

# Resume an interrupted run instead of starting a new one (README "Resuming a run"). Give the
# run's id, its folder under results/, and a decision step it saved a checkpoint at and finished
# evaluating. It keeps the run's id and folders, skips the work already done, finishes an
# introspection branch at that step that never did, and trains on from the next step. Only
# compute, gates, training_steps and later introspection points may differ from the run's own
# settings; anything else is refused. None starts a new run. For the run that ran out of memory:
#   SETTINGS_FILE = "configs/paper_minimum_threshold_8b_16gb_resume.json"
#   RESUME_RUN_ID = "20261031-061400_paper_minimum_threshold_Qwen3-8B"
#   RESUME_FROM_STEP = 1500
RESUME_RUN_ID = None
RESUME_FROM_STEP = None

# The branch of the repo to run. On a fresh machine it is cloned; an existing clone is moved
# onto it and brought up to date, so the code is always this branch's latest commit.
BRANCH = "bootstrap-ci"

# Where finished runs are saved so they survive the machine. Must be a Hub repo you
# can write to; the owner must match the account your token belongs to.
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"

# Settings files available:
#   configs/default.json             THE DEFAULT: Plunkett on Qwen3-0.6B, 4-bit, fp16 (Colab T4)
#   configs/atkinson.json            same but 3000 steps, no report training
#   configs/linear-4b_20gb.json      Qwen3-4B, bf16, unquantized, gradient checkpointing
#   configs/linear-8b_20gb.json      Qwen3-8B, bf16, 4-bit, gradient checkpointing
#   configs/minimum_threshold_8b.json        the minimum threshold rule at Qwen3-8B, 3000 steps, answers A/B/C
#   configs/paper_minimum_threshold_8b.json  the paper's minimum threshold design at Qwen3-8B, answers A/B/C
#   configs/paper_minimum_threshold_0.6b.json the same at Qwen3-0.6B, only the model changed
#   configs/paper_minimum_threshold_8b_16gb.json the paper design on a 16 GB card (the 20261031 run)
#   configs/minimum_threshold_1500_steps.json   the same, 1500 steps over 100 trials per persona,
#                                               introspection toward hidden and recovered targets
#
# The two linear-*_20gb files are sized for a 20 GB card, which is what the GPU check
# below reports on a MIG slice of a Hopper card.
#
# Introspection, set in the settings file under model_hyperparameters (README "Settings"):
#   introspection_at_steps    the decision steps stage two branches from. A list, [1500, 2500, 3000],
#                             or a range, {"start": 1000, "end": 3000, "every": 500}, which is
#                             1000, 1500, ..., 3000. null: the last decision step only
#   introspection_steps       how long each branch trains (its end point), per fold
#   introspection_batch_size  examples per introspection step
#   introspection_training    false switches stage two off altogether
#   training_steps            the last decision step; no branch point may lie past it
#   introspection_targets     what each branch trains toward, one branch each: hidden (the persona's
#                             own rule), recovered (the rule read from the model's own choices),
#                             shuffled (another persona's: format without content), random
#   introspection_remeasure_decisions  ask the held-out personas' decisions again after each branch
#   lr_schedule               null (constant) or {"type": "linear"/"cosine", "warmup_steps": W,
#                             "final_fraction": f}
#
# Which questions are asked and trained on: report_schema.questions in the settings file, each
# {"ask": true/false, "train": true/false}; every question is a folder under questions/ (README "Questions").


## Set up the machine

In [9]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, use USE_TEST_CONFIG = True"

Mon Nov  9 21:47:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.88                 KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   42C    P5             17W /  300W |    1241MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

code: branch bootstrap-ci, commit 81b5046 Maximum threshold experiment, one-persona runs and cut sweep
repo ready in C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments


In [11]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Scripts\python.exe -m pip install --upgrade pip


In [12]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")


kernel python: C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Scripts\python.exe
all imports available


In [13]:
# Confirm the GPU matches what the settings assume. bf16 needs Ampere or newer:
# the 20gb configs ask for it and the loader refuses rather than falling back.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 30:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB, not a whole card.")
        print("Use the *_20gb configs for 4B and 8B. 14B will not fit here.")
    elif vram < 60:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB. The *_20gb configs fit with room to spare.")
else:
    print("no CUDA device: only USE_TEST_CONFIG = True will run here")

torch 2.8.0+cu128
gpu NVIDIA GeForce RTX 5070 Ti
vram GB 17.1
bf16 supported True

This is a MIG slice of about 17 GB, not a whole card.
Use the *_20gb configs for 4B and 8B. 14B will not fit here.


In [14]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only for the save cells; the public Qwen3 weights do not
# require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

owner = HUB_REPO.split("/")[0]
who = None
for attempt in range(3):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if who is None:
    print("no token: the run works, but saving refuses and nothing survives the machine")
else:
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Set HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\" in the first cell and re-run it.")


Hugging Face WRITE token (blank to skip):  ········


token belongs to: adamchanna
HUB_REPO owner 'adamchanna' matches. Saving will work.


## Run the experiment

On a rented box, go in this order and save after each one. Starting small
exercises the CUDA path, the estimator, the report parser and the results writer in
a few minutes, which is the cheapest place to find a broken setting.

    default  ->  linear-4b_20gb  ->  linear-8b_20gb

For the minimum threshold experiments, the 0.6B run first, then 8B:

    paper_minimum_threshold_0.6b  ->  paper_minimum_threshold_8b

The cell prints the plan before training starts: the rule, the model, the decision steps,
where introspection branches and for how long. Check it before walking away.

In [15]:
from src.config import load
from src.pipeline import run

from src.config import estimation_counts, eval_batch_size, introspection_points

cfg = load(None if USE_TEST_CONFIG else SETTINGS_FILE, use_test=USE_TEST_CONFIG)
mh, rule = cfg["model_hyperparameters"], cfg["decision_rule"]
points = introspection_points(cfg)
print(f"settings      {cfg['_source']}")
print(f"rule          {rule['type']}, {rule['option_count']} options, data {cfg['model_training']['data_dir']}")
print(f"model         {mh['model_name']}, {mh['quantization']}, {cfg['compute']['mixed_precision']} on {cfg['compute']['device']}")
print(f"decisions     {mh['training_steps']} steps, a checkpoint every {mh['checkpoint_every']}, "
      f"baseline {'on' if mh['baseline_before_training'] else 'off'}")
print(f"introspection " + (f"from decision steps {points}, {mh['introspection_steps']} steps per fold, "
                           f"targets {mh['introspection_targets']}"
                           + (", decisions re-measured after" if mh["introspection_remeasure_decisions"] else "")
                           if points else "off"))
print(f"learning rate {mh['learning_rate']}, schedule {mh['lr_schedule'] or 'constant'}")
questions = cfg["report_schema"].get("questions")
if questions:
    print("questions     " + ", ".join(f"{q} ({'asked, trained' if f.get('train', True) else 'asked only'})"
                                          for q, f in questions.items() if f.get("ask", True)))
print(f"estimation    {estimation_counts(cfg)} trials per persona")
print(f"eval batches  decision {eval_batch_size(cfg, 'decision')}, introspection {eval_batch_size(cfg, 'introspection')}")

RESUME = {"run_id": RESUME_RUN_ID, "from_step": RESUME_FROM_STEP} if RESUME_RUN_ID else None
print(f"resume        " + (f"{RESUME_RUN_ID} from decision step {RESUME_FROM_STEP}" if RESUME else "off: a new run"))

rows = run(cfg, resume=RESUME)

settings      configs\one_persona_minimum_threshold_b.json <- configs\one_persona_minimum_threshold.json <- configs\minimum_threshold_1500_steps.json <- configs\paper_minimum_threshold_8b_16gb.json <- configs\default.json
rule          paper_minimum_threshold, 3 options, data data/one_persona_minimum_threshold_b/
model         Qwen/Qwen3-8B, 4bit, bf16 on cuda
decisions     3000 steps, a checkpoint every 250, baseline on
introspection off
learning rate 0.0002, schedule constant
questions     main (asked, trained), screen_index (asked, trained), screen_cut_units (asked, trained), main_report_units (asked, trained)
estimation    [1000] trials per persona
eval batches  decision 4, introspection 1
resume        off: a new run
compute backend: local, device: cuda
validation: 1000 natural trials, not the folder's quota trials | cut_agrees 241, cut_overturns 250, neither 249, weights 260
data: data/one_persona_minimum_threshold_b/ | 1 personas | 5000 training and 1000 validation examples | ru

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\transformers\modeling_utils.py:6231: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  _ = torch.empty(byte_count // factor, dtype=torch.float16, device=device, requires_grad=False)
Loading checkpoint shards: 100%|██████████| 5/5 [00:13<00:00,  2.65s/it]


model: Qwen/Qwen3-8B | quantization=4bit | device=cuda | dtype=bfloat16
gradient checkpointing: on
finetune: lora rank 8 alpha 8 on ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']; 21,823,488 trainable parameters
base model: the untrained model, scored before decision training
evaluating base_model_step-0 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 0/1, cut too for 0/1
  decisions: 1000 rows, accuracy 0.392, 84s
    generating: 20/50 replies, 28s
    generating: 40/50 replies, 56s
    generating: 50/50 replies, 72s
  reports [paper_minimum_threshold/main]: 50/50 parsed, 156s
    generating: 20/50 replies, 7s
    generating: 40/50 replies, 13s
    generating: 50/50 replies, 17s
  reports [paper_minimum_threshold/screen_index]: 50/50 parsed, 173s
    generating: 20/50 replies, 13s
    generating: 40/50 replies, 25s
    generating: 50/50 replies, 32s
  reports [paper_minimum_threshold/screen_cut_units]: 47/50 parsed, 205s

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  step 50/3000 loss 0.6760 acc 0.478 [mean of steps 1-50] | 132s
  step 100/3000 loss 0.3775 acc 0.666 [mean of steps 51-100] | 265s
  step 150/3000 loss 0.2034 acc 0.866 [mean of steps 101-150] | 397s
  step 200/3000 loss 0.1632 acc 0.888 [mean of steps 151-200] | 530s
  step 250/3000 loss 0.1163 acc 0.908 [mean of steps 201-250] | val loss 0.0744 val acc 0.944 weights 0.892 neither 1.000 cut_agrees 0.979 cut_overturns 0.908 (1000 trials, 71s) | largest gradient 28.79, 104 steps clipped at 1.0 | saved checkpoints\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b\decision\step-250 | 734s
evaluating decision_step-250 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 1/1, cut too for 1/1
  decisions: 1000 rows, accuracy 0.958, 84s
    generating: 20/50 replies, 31s
    generating: 40/50 replies, 62s
    generating: 50/50 replies, 80s
  reports [paper_minimum_threshold/main]: 50/50 parsed, 164s
    generating: 20/50 replies, 5s
    generating: 40/5

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


... (19 outputs left out)

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  step 2800/3000 loss 0.0382 acc 0.974 [mean of steps 2751-2800] | 11544s
  step 2850/3000 loss 0.0404 acc 0.972 [mean of steps 2801-2850] | 11675s
  step 2900/3000 loss 0.0399 acc 0.976 [mean of steps 2851-2900] | 11806s
  step 2950/3000 loss 0.0226 acc 0.984 [mean of steps 2901-2950] | 11937s
  step 3000/3000 loss 0.0447 acc 0.978 [mean of steps 2951-3000] | val loss 0.0147 val acc 0.989 weights 0.958 neither 1.000 cut_agrees 1.000 cut_overturns 1.000 (1000 trials, 70s) | largest gradient 4.54, 29 steps clipped at 1.0 | saved checkpoints\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b\decision\step-3000 | 12139s
evaluating decision_step-3000 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 1/1, cut too for 1/1
  decisions: 1000 rows, accuracy 0.989, 83s
    generating: 20/50 replies, 8s
    generating: 40/50 replies, 20s
    generating: 50/50 replies, 21s
  reports [paper_minimum_threshold/main]: 3/50 parsed, 105s
    generating: 20/50 repl

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\src\plots.py:357: DtypeWarning: Columns (19: introspection_target) have mixed types. Specify dtype option on import or set low_memory=False.
  made.append(plot_training(pd.read_csv(log_path), run_id, out_dir))


plots: results\plots\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b_training.png, results\plots\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b_checkpoints.png, results\plots\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b_scatter_paper_minimum_threshold_main.png, results\plots\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b_scatter_paper_minimum_threshold_main_report_units-main.png

results for 20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b (12961s), appended to results\results.csv:
     stage introspection_target  introspection_fold introspection_from_step  checkpoint_step           report_schema      report_batch         block  estimation_trials  decision_accuracy  recovered_vs_hidden  faithfulness  shuffled_faithfulness  faithfulness_shuffled_p  chance  parse_rate  n_personas_scored  hidden_vs_reported  shuffled_hidden_vs_reported
base_model                                        0                                        0 pap

## Save it off the machine

Results are small and worth pushing often. Adapters are tens of megabytes, so pass
`checkpoints=False` for a quick repeat push mid-run.

In [16]:
from src.persist import push_run

push_run(rows[-1]["run_id"], HUB_REPO)

20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b_checkpoints.png: 100%|██████████| 207k/207k [00:00<00:00, 404kB/s]  
Upload 13 LFS files:   0%|          | 0/13 [00:00<?, ?it/s]
adapter_model.safetensors:   0%|          | 0.00/87.4M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/87.4M [00:00<?, ?B/s]


adapter_model.safetensors:   0%|          | 0.00/87.4M [00:00<?, ?B/s]




adapter_model.safetensors:   0%|          | 0.00/87.4M [00:00<?, ?B/s]



adapter_model.safetensors:   0%|          | 0.00/87.4M [00:00<?, ?B/s]


adapter_model.safetensors:   0%|          | 16.4k/87.4M [00:00<26:57, 54.0kB/s]

adapter_model.safetensors:   0%|          | 16.4k/87.4M [00:00<27:45, 52.4kB/s]
adapter_model.safetensors:   0%|          | 16.4k/87.4M [00:00<28:17, 51.5kB/s]



adapter_model.safetensors:   0%|          | 16.4k/87.4M [00:00<27:29, 53.0kB/s]




adapter_model.safetensors:   0%|          | 16.4k/87.4M [00:00<33:05, 44.0kB/s]



adapter_model.safetensors:   0%|  

pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b
  results\plots
  checkpoints\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b


['results\\results.csv',
 'results\\train_log.csv',
 'results\\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b',
 'results\\plots',
 'checkpoints\\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b']

In [17]:
# Before you walk away: sweep everything still on disk, including any run that was
# interrupted part way. Results are appended as a run goes, so a partial run is
# still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-202425_paper_minimum_threshold_Qwen3-0.6B
  results\plots


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-204411_paper_minimum_threshold_Qwen3-0.6B
  results\plots


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-211646_paper_minimum_threshold_Qwen3-0.6B
  results\plots


... (60 outputs left out)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b
  results\plots
  checkpoints\20261109-214726_paper_minimum_threshold_Qwen3-8B_persona-b


No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\queue
  results\plots


## Cut training

The cut-training runs (branched from the one-persona run's step 1500, with a sweep of how many training trials the cut decides) have their own notebook: **prior_experiment_training.ipynb**. It builds the runs, queues them one after another in the background, shows their status and results side by side, and carries finished runs on further.

## Persona B control

The one-persona run (persona A, `20261106-152354`) reported weights scoring 0.65 against the true ones at steps
1500 to 2750. The cross-persona check (`tools/cross_persona_control.py`) found that number is explained by a
constant "all -100" reply (it scores 0.61 for free, since A's weights are mostly negative) and by what the
untrained model already believes about the attribute names (its step-0 report already ranks the attributes
nearly right). Persona B is the control that settles it: **the same persona, prompt, attributes, caffeine screen
and the very same 5000 training trials, with different weights**, so 52% of the A/B answers flip.

| | caffeine_content | leaf_size | oxidation_level | steeping_time | resteep_potential |
|---|---|---|---|---|---|
| A (run 20261106-152354) | 0 (screened) | -83 | -38 | -100 | 31 |
| B (this run) | 0 (screened) | 100 | 20 | -50 | -70 |

B's weights sum to 0, so any constant reply scores 0 against them, and they run against the untrained model's
ranking. If B's reports match B's weights better than A's (and A's match A's better than B's), the reports carry
what decision training put there. If B's reports look like A's, it was priors.

Settings: `configs/one_persona_minimum_threshold_b.json` (A's settings plus gradient clipping 1.0, 3000 steps,
a checkpoint every 250, optimizer state kept for the last one only). Data: `data/one_persona_minimum_threshold_b/`
(built and verified). About 5 to 6 hours on the 5070 Ti.

**Run order:** the check cell, then the start cell, then walk away. The run is a separate background process: it
does not need this notebook, its kernel or Jupyter, and it keeps Windows from idle sleep while it runs (a closed
lid, Sleep, logging off or an update restart still end it). Come back to the status cell, then results, then save.
The cells below need nothing from the cells above, except save, which needs `HUB_REPO` (settings cell) and the
token (credentials cell).

In [ ]:
# Persona B, step 1: check everything before starting. Changes nothing, and touches no GPU memory.
import json, shutil, subprocess, sys
from pathlib import Path

from src.config import resolve
from src.experiments import describe, disk_needed_gb, load_experiment
from src.rules.base import check_manifest

PERSONA_B_CONFIG = "configs/one_persona_minimum_threshold_b.json"
PERSONA_B_QUEUE = "persona_b"                                        # its state: results/queue/persona_b/
PERSONA_A_RUN = "20261106-152354_paper_minimum_threshold_Qwen3-8B"   # the run it is compared with
PERSONA_B_CHECKED = False

cfg_b = load_experiment(PERSONA_B_CONFIG)
mh_b, data_b = cfg_b["model_hyperparameters"], Path(cfg_b["model_training"]["data_dir"])
describe(cfg_b)
check_manifest(str(data_b), resolve(cfg_b).rule, cfg_b)           # raises if the folder disagrees with the settings
print(f"data          {data_b} built, manifest matches")

# the same trials as persona A, with B's answers
read = lambda p: {json.loads(l)["messages"][1]["content"]: json.loads(l)["messages"][2]["content"]
                  for l in open(p, encoding="utf-8")}
trials_a = read("data/one_persona_minimum_threshold/instill_1_prefs.jsonl")
trials_b = read(data_b / "instill_1_prefs.jsonl")
assert trials_a.keys() == trials_b.keys(), "persona B's training prompts are not persona A's"
flipped = sum(trials_a[k] != trials_b[k] for k in trials_a)
print(f"trials        {len(trials_b)}, the same prompts as A; {flipped} answers differ from A's")
assert Path("results", PERSONA_A_RUN, "settings.json").exists(), f"results/{PERSONA_A_RUN} is missing: it is the comparison"
assert mh_b["max_grad_norm"] == 1.0 and mh_b["training_steps"] == 3000 and cfg_b["compute"]["device"] == "cuda"

need, free = disk_needed_gb(cfg_b), shutil.disk_usage(".").free / 1024 ** 3
print(f"disk          about {need:.1f} GB needed, {free:.1f} GB free")
assert free > need, "not enough disk"

used, total = (int(x) for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")[:2])
print(f"gpu           {used} of {total} MiB in use" + ("" if total - used > 12 * 1024 else
      "  <- the queue waits until 12 GB are free; close whatever holds the card (another run, a kernel that loaded the model)"))

status_file = Path("results/queue", PERSONA_B_QUEUE, "status.json")
if status_file.exists():
    for r in json.loads(status_file.read_text(encoding="utf-8"))["runs"]:
        print(f"queue         already has {r['config']}: {r['status']} {r.get('run_id') or ''}")
PERSONA_B_CHECKED = True
print("\nall checks passed: run the next cell to start")

In [ ]:
# Persona B, step 2: start the run as its own hidden background process, then walk away.
# Started through Windows (WMI) rather than as a child of this kernel, so closing the notebook, the kernel or
# Jupyter leaves it running, and with no console window that a click could close. scripts/run_awake.py holds off
# idle sleep while it runs; scripts/run_queue.py records progress in results/queue/persona_b/ and refuses to start
# a second copy. Running this cell again after the run finished does nothing (the queue skips what is done).
assert globals().get("PERSONA_B_CHECKED"), "run the check cell above first"
root = Path.cwd().resolve()
command = (f'"{sys.executable}" "{root / "scripts" / "run_awake.py"}" "{root / "scripts" / "run_queue.py"}" '
           f'--name {PERSONA_B_QUEUE} {PERSONA_B_CONFIG}')
ps = ("$si = New-CimInstance -ClassName Win32_ProcessStartup -ClientOnly -Property @{ShowWindow=[uint16]0}; "
      f"$r = Invoke-CimMethod -ClassName Win32_Process -MethodName Create -Arguments @{{CommandLine='{command}'; "
      f"CurrentDirectory='{root}'; ProcessStartupInformation=$si}}; " + '"$($r.ReturnValue) $($r.ProcessId)"')
result = subprocess.run(["powershell", "-NoProfile", "-NonInteractive", "-Command", ps], capture_output=True, text=True)
code_, pid = (result.stdout.split() + ["?", "?"])[:2]
if code_ != "0":
    raise RuntimeError(f"Windows refused to start it (return value {code_}): {result.stderr or result.stdout}")
print(f"started, process {pid}. Log: results/queue/{PERSONA_B_QUEUE}/queue.log and the run's own log there.")
print("Give it a minute, then run the status cell: it should say running.")

In [ ]:
# Persona B, step 3: status. Run it any time; it only reads.
import json, re, subprocess, sys
from pathlib import Path
PERSONA_B_QUEUE = globals().get("PERSONA_B_QUEUE", "persona_b")
qdir = Path("results/queue", PERSONA_B_QUEUE)
print(subprocess.run([sys.executable, "scripts/run_queue.py", "--name", PERSONA_B_QUEUE, "--status"],
                     capture_output=True, text=True).stdout)
logs = sorted(qdir.glob("*.log"), key=lambda p: p.stat().st_mtime)
for log in logs[-2:]:
    text = log.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"--- last lines of {log}")
    print("\n".join(text[-12:]))

In [ ]:
# Persona B, step 4: results, once the run is done (or part way: it reads whatever checkpoints are scored).
# The two-way test on the "main" question: each persona's mean report scored against its own weights and the
# other persona's. "no tmpl" drops the constant all-equal replies first; "centred" removes each vector's average.
import importlib.util, json, subprocess, sys
from pathlib import Path
import pandas as pd

PERSONA_B_QUEUE = globals().get("PERSONA_B_QUEUE", "persona_b")
PERSONA_A_RUN = globals().get("PERSONA_A_RUN", "20261106-152354_paper_minimum_threshold_Qwen3-8B")
status_file = Path("results/queue", PERSONA_B_QUEUE, "status.json")
runs = json.loads(status_file.read_text(encoding="utf-8"))["runs"] if status_file.exists() else []
if runs and runs[0].get("run_id"):
    PERSONA_B_RUN = runs[0]["run_id"]
else:       # started from the main run cell instead of the queue: the newest persona-b run folder
    PERSONA_B_RUN = sorted(p.name for p in Path("results").glob("*_persona-b"))[-1]
    runs = [{"run_id": PERSONA_B_RUN, "status": "started outside the queue"}]
print(f"persona B run {PERSONA_B_RUN} ({runs[0]['status']}), persona A run {PERSONA_A_RUN}\n")

print(subprocess.run([sys.executable, "scripts/faithfulness_curve.py", PERSONA_B_RUN, "--batch", "main"],
                     capture_output=True, text=True).stdout)

spec = importlib.util.spec_from_file_location("cross_persona_control", "tools/cross_persona_control.py")
cpc = importlib.util.module_from_spec(spec); spec.loader.exec_module(cpc)
b = cpc.control(Path("results", PERSONA_B_RUN), "main", "main", against=Path("results", PERSONA_A_RUN))
a = cpc.control(Path("results", PERSONA_A_RUN), "main", "main", against=Path("results", PERSONA_B_RUN))
keep = ["checkpoint_step", "template_share", "own", "against", "no_template", "against_no_template",
        "own_centred", "against_centred"]
table = a[keep].merge(b[keep], on="checkpoint_step", suffixes=("_A", "_B"))
table.columns = pd.MultiIndex.from_tuples(
    [("", "step")] + [(f"report {p}", h) for p in "AB" for h in
                      ["all-equal share", "vs own", "vs other", "vs own, no tmpl", "vs other, no tmpl", "own centred", "other centred"]])
with pd.option_context("display.width", 250, "display.float_format", "{:.2f}".format):
    print(table.to_string(index=False))
print("\nSignal from training: report B scores clearly higher vs own (B) than vs other (A) once decisions are learned,")
print("including with no tmpl, while step 0 does not. B's weights sum to 0, so the all-equal reply scores 0 against them.")
print(f"Full tables: results/{PERSONA_B_RUN}/cross_persona_control_vs_{PERSONA_A_RUN}.csv and the A-side file.")

In [ ]:
# Persona B, step 5: save the run to the Hub (needs HUB_REPO from the settings cell and the token from the
# credentials cell). Results and checkpoints go to the same repo as every other run.
from src.persist import push_run
push_run(PERSONA_B_RUN, HUB_REPO)

In [ ]:
# Persona B, only if needed: stop the run NOW. Checkpoints already saved stay on disk; the step in progress is lost.
# Afterwards the status file may still say "running". To carry on from the last saved checkpoint instead of
# starting over: python scripts/run_one_persona.py --settings configs/one_persona_minimum_threshold_b.json --resume <run_id> <step>
import subprocess
from pathlib import Path
lock = Path("results/queue", globals().get("PERSONA_B_QUEUE", "persona_b"), "queue.lock")
if lock.exists():
    pid = lock.read_text().split()[0]
    print(subprocess.run(["taskkill", "/PID", pid, "/T", "/F"], capture_output=True, text=True).stdout)
else:
    print("no queue lock: nothing running")

## No-trial questions

**What changed.** The report questions used to end with a decision trial (two options, no C). The model answered the
trial instead of reporting its rule: "all -100" whenever both options failed the caffeine cut, a cut copied off the
shown numbers, and drift into bare A/B answers. Five new questions show **no trial, only the persona line**
("Imagine you are Djedefre. Which loose leaf tea would you prefer?"); the cut questions list each dimension's
units and range instead. Each is asked once per checkpoint (temperature 0 gives one answer).

| question | asks | right answer (persona A / B) |
|---|---|---|
| `main_threshold_met` | the weights once an option meets the caffeine minimum | A: 0, -83, -38, -100, 31 / B: 0, 100, 20, -50, -70 |
| `main_threshold_not_met` | the weights when an option falls below it | not agreed yet: read the replies |
| `screen_index_no_trial` | which dimension carried the minimum | 1 (caffeine_content) |
| `screen_cut_units_no_trial` | that dimension and its minimum in its units | 1, 35 mg |
| `main_report_units_no_trial` | weights + dimension + minimum in one reply | as above |

The old four questions (`main`, `screen_index`, `screen_cut_units`, `main_report_units`) are still asked too, so
these runs compare directly with runs A and B. Nothing is trained on the questions (introspection off).

**The runs.** Two decision-training runs from scratch on Qwen3-8B, one after the other: persona A
(`configs/one_persona_no_trial_a.json`) then persona B (`configs/one_persona_no_trial_b.json`). Same data, seeds and
learning rate as runs A and B, gradient clipping 1.0, 3000 steps, a checkpoint every 250. About 5 to 6 hours each.

**Before you start:** if this kernel loaded a model (the main run cell, e.g. run B), restart it first
(Kernel > Restart Kernel): it holds the GPU and the runs would wait for it. Do **not** use the main run cell for
these runs; use the start cell below (it keeps the PC awake and survives closing the notebook).

**Order:** check, start, walk away. Then status, results, save.

In [ ]:
# No-trial questions, step 1: check everything before starting. Changes nothing, and touches no GPU memory.
import json, shutil, subprocess, sys
from pathlib import Path

from src import data as D
from src.config import resolve
from src.experiments import describe, disk_needed_gb, load_experiment
from src.rules.base import check_manifest

NO_TRIAL_CONFIGS = ["configs/one_persona_no_trial_a.json", "configs/one_persona_no_trial_b.json"]   # run in this order
NO_TRIAL_QUEUE = "no_trial"                                                                          # results/queue/no_trial/
NO_TRIAL_CHECKED = False

need = 0.0
for path in NO_TRIAL_CONFIGS:
    cfg_nt = load_experiment(path)
    comps_nt = resolve(cfg_nt)
    describe(cfg_nt)
    check_manifest(cfg_nt["model_training"]["data_dir"], comps_nt.rule, cfg_nt)
    data_nt = D.load_data(cfg_nt, comps_nt.rule)
    batches = comps_nt.schemas[0].batches_for(data_nt.personas[0])
    no_trial = [b.name for b in batches if not b.show_trial]
    assert len(no_trial) == 5, f"{path}: expected 5 no-trial questions, got {no_trial}"
    mh_nt = cfg_nt["model_hyperparameters"]
    assert mh_nt["max_grad_norm"] == 1.0 and mh_nt["training_steps"] == 3000 and cfg_nt["compute"]["device"] == "cuda"
    print(f"   persona weights {data_nt.latents[0][:5].tolist()}; no-trial questions {no_trial}")
    print("   example, as the model sees it:\n      " +
          D.report_prompt(batches[[b.name for b in batches].index("main_threshold_met")], D.Trial(data_nt.personas[0], __import__("random").Random(0))).replace("\n", "\n      ") + "\n")
    need += disk_needed_gb(cfg_nt)

free = shutil.disk_usage(".").free / 1024 ** 3
print(f"disk          about {need:.1f} GB needed for both runs, {free:.1f} GB free")
assert free > need, "not enough disk"
used, total = (int(x) for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")[:2])
print(f"gpu           {used} of {total} MiB in use" + ("" if total - used > 12 * 1024 else
      "  <- STOP: something holds the GPU. If this kernel ran a model, Kernel > Restart Kernel, then run this cell again"))
assert total - used > 12 * 1024, "the GPU is busy; free it first (see above)"
status_file = Path("results/queue", NO_TRIAL_QUEUE, "status.json")
if status_file.exists():
    for r in json.loads(status_file.read_text(encoding="utf-8"))["runs"]:
        print(f"queue         already has {r['config']}: {r['status']} {r.get('run_id') or ''}")
NO_TRIAL_CHECKED = True
print("\nall checks passed: run the next cell to start")

In [ ]:
# No-trial questions, step 2: start both runs (A, then B) as one hidden background process, then walk away.
# Started through Windows (WMI), not as a child of this kernel: closing the notebook, the kernel or Jupyter leaves it
# running, with no console window to close by accident. scripts/run_awake.py keeps the PC from idle sleep while it runs
# (closing a laptop lid, Sleep, logging off or an update restart still stop it). scripts/run_queue.py runs the configs
# one after another, records progress in results/queue/no_trial/, and refuses to start twice. Run again after a crash
# or reboot: it skips what is done.
assert globals().get("NO_TRIAL_CHECKED"), "run the check cell above first"
root = Path.cwd().resolve()
command = (f'"{sys.executable}" "{root / "scripts" / "run_awake.py"}" "{root / "scripts" / "run_queue.py"}" '
           f'--name {NO_TRIAL_QUEUE} ' + " ".join(NO_TRIAL_CONFIGS))
ps = ("$si = New-CimInstance -ClassName Win32_ProcessStartup -ClientOnly -Property @{ShowWindow=[uint16]0}; "
      f"$r = Invoke-CimMethod -ClassName Win32_Process -MethodName Create -Arguments @{{CommandLine='{command}'; "
      f"CurrentDirectory='{root}'; ProcessStartupInformation=$si}}; " + '"$($r.ReturnValue) $($r.ProcessId)"')
result = subprocess.run(["powershell", "-NoProfile", "-NonInteractive", "-Command", ps], capture_output=True, text=True)
code_, pid = (result.stdout.split() + ["?", "?"])[:2]
if code_ != "0":
    raise RuntimeError(f"Windows refused to start it (return value {code_}): {result.stderr or result.stdout}")
print(f"started, process {pid}. Logs: results/queue/{NO_TRIAL_QUEUE}/")
print("Give it a minute, then run the status cell: it should say running.")

In [ ]:
# No-trial questions, step 3: status. Run it any time; it only reads.
import json, subprocess, sys
from pathlib import Path
NO_TRIAL_QUEUE = globals().get("NO_TRIAL_QUEUE", "no_trial")
qdir = Path("results/queue", NO_TRIAL_QUEUE)
print(subprocess.run([sys.executable, "scripts/run_queue.py", "--name", NO_TRIAL_QUEUE, "--status"],
                     capture_output=True, text=True).stdout)
for log in sorted(qdir.glob("*.log"), key=lambda p: p.stat().st_mtime)[-2:]:
    print(f"--- last lines of {log}")
    print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]))

In [ ]:
# No-trial questions, step 4: results, per run and checkpoint (works part way through a run too).
# For each checkpoint: decision accuracy; the old trial-shown main question's score; and the five no-trial replies,
# word for word, scored where there is a right answer. Cosine on the four weighted dimensions leaves out
# caffeine_content, whose weight is 0 once the minimum is met.
import json, re
from pathlib import Path
import numpy as np
import pandas as pd

NO_TRIAL_QUEUE = globals().get("NO_TRIAL_QUEUE", "no_trial")
TRUE = {"one_persona_no_trial_a": [0, -83, -38, -100, 31], "one_persona_no_trial_b": [0, 100, 20, -50, -70]}
NAMES = ["caffeine_content", "leaf_size", "oxidation_level", "steeping_time", "resteep_potential"]
cos = lambda a, b: float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))) if np.linalg.norm(a) and np.linalg.norm(b) else float("nan")

def reply(rows, batch):
    found = [r for r in rows if r["batch"] == batch]
    return found[0]["reply"].strip() if found else None

def weights(text):
    try:
        j = json.loads(text)
        return np.array([float(j[n]) for n in NAMES])
    except Exception:
        return None

results = pd.read_csv("results/results.csv", low_memory=False)
status = json.loads(Path("results/queue", NO_TRIAL_QUEUE, "status.json").read_text(encoding="utf-8"))["runs"]
NO_TRIAL_RUNS = [r["run_id"] for r in status if r.get("run_id")]
for r in status:
    run_id, true = r.get("run_id"), np.array(TRUE[Path(r["config"]).stem], float)
    if not run_id:
        print(f"{r['config']}: {r['status']}, not started yet\n"); continue
    print(f"===== {run_id} ({r['status']})   true weights {true.astype(int).tolist()}, screen 1, minimum 35 mg")
    rows_out = []
    files = sorted(Path("results", run_id).glob("reports_raw_*.jsonl"),
                   key=lambda p: 0 if "base_model" in p.name else int(re.findall(r"(\d+)", p.name)[-1]))
    for f in files:
        step = 0 if "base_model" in f.name else int(re.findall(r"(\d+)", f.name)[-1])
        rows = [json.loads(l) for l in open(f, encoding="utf-8")]
        res = results[(results.run_id == run_id) & (results.checkpoint_step == step)]
        acc = res.decision_accuracy.dropna().iloc[0] if res.decision_accuracy.notna().any() else np.nan
        old = res[(res.report_batch == "main") & (res.block == "main")]
        met, notmet = weights(reply(rows, "main_threshold_met") or ""), weights(reply(rows, "main_threshold_not_met") or "")
        cut = reply(rows, "screen_cut_units_no_trial") or ""
        m = re.search(r'"screened_dimension":\s*(\d+).*?"minimum_value":\s*(-?[\d.]+)', cut)
        idx = re.search(r'"screened_dimension":\s*(\d+)', reply(rows, "screen_index_no_trial") or "")
        rows_out.append({
            "step": step, "decisions": acc,
            "old main (trial shown)": old.hidden_vs_reported.iloc[0] if len(old) else np.nan,
            "met: reply": met.astype(int).tolist() if met is not None else (reply(rows, "main_threshold_met") or "")[:40],
            "met: cos 5": cos(met, true) if met is not None else np.nan,
            "met: cos 4": cos(met[1:], true[1:]) if met is not None else np.nan,
            "not met: reply": notmet.astype(int).tolist() if notmet is not None else (reply(rows, "main_threshold_not_met") or "")[:40],
            "screen (1)": int(idx.group(1)) if idx else (reply(rows, "screen_index_no_trial") or "")[:12],
            "cut (1, 35 mg)": f"{m.group(1)}, {float(m.group(2)):g}" if m else cut[:24],
        })
    with pd.option_context("display.width", 250, "display.max_colwidth", 45, "display.float_format", "{:.2f}".format):
        print(pd.DataFrame(rows_out).to_string(index=False), "\n")
print("Every reply word for word: results/<run>/reports_raw_<step>.jsonl. All scores: results/results.csv,"
      " report_batch = the question's name.")

In [ ]:
# No-trial questions, step 5: save both runs to the Hub (needs HUB_REPO from the settings cell and the token from
# the credentials cell).
from src.persist import push_run
for run_id in NO_TRIAL_RUNS:
    push_run(run_id, HUB_REPO)

In [ ]:
# No-trial questions, only if needed: stop NOW (the run in progress and the rest of the queue). Saved checkpoints stay.
# To end after the run in progress instead: python scripts/run_queue.py --name no_trial --stop
import subprocess
from pathlib import Path
lock = Path("results/queue", globals().get("NO_TRIAL_QUEUE", "no_trial"), "queue.lock")
if lock.exists():
    pid = lock.read_text().split()[0]
    print(subprocess.run(["taskkill", "/PID", pid, "/T", "/F"], capture_output=True, text=True).stdout)
else:
    print("no queue lock: nothing running")